## Setup

Set real_time to false if you want to read from file

In [13]:
import serial
import csv
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy.special import gammaincc
from scipy.stats import norm
import numpy.fft as fft

PORT = "COM8"
BAUD = 230400

load_from_file = False
load_file_path = Path("data/06_25_2026/run_15_13_02.csv")
sensors = ['BPW34_1', 'BPW34_2', 'Inano_1', 'Inano_2']

adc_max = 16383 #2^14 - 1
v_ref = 5
SAMPLES_PER_PACKET = 50
payload_dtype = np.dtype([
    ('timestamp', '<u4'),
    ('BPW34_1', '<u2'),
    ('BPW34_2', '<u2'),
    ('Inano_1', '<u2'),
    ('Inano_2', '<u2')
])



BYTES_PER_SAMPLE = payload_dtype.itemsize * SAMPLES_PER_PACKET


def fir_lowpass_filter(data, cutoff_hz, fs, num_taps=101):
    b = signal.firwin(num_taps, cutoff=cutoff_hz, fs=fs, window='hamming')
    filtered_data = np.asarray(signal.lfilter(b, 1.0, data))
    return filtered_data, b


## Saves realtime data to file


In [ ]:
ser = serial.Serial(PORT, BAUD, timeout=1)
NUM_SAMPLES_TO_SAVE = 1000   # set to None to run until interrupted

now = datetime.now()
savefile_path = Path(f"data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}.csv")
savefile_path.parent.mkdir(parents=True, exist_ok=True)

header_written = False
sample_count = 0


##read packets of data from the serial port and save it to a CSV file
try:
    while NUM_SAMPLES_TO_SAVE is None or sample_count < NUM_SAMPLES_TO_SAVE:
        if ser.read(1) == b'\xAA' and ser.read(1) == b'\xBB':
            raw_data = ser.read(BYTES_PER_SAMPLE) 
            if len(raw_data) == BYTES_PER_SAMPLE:
                data = np.frombuffer(raw_data, dtype=payload_dtype)
                mean_delta_us = np.mean(np.diff(data['timestamp']))
                if mean_delta_us > 0:
                    sampling_rate = 1.0 / (mean_delta_us * 1e-6)
                    # write all samples in this packet to file
                    df = pd.DataFrame(data)
                    df.to_csv(savefile_path, mode='a', header=not header_written, index=False)
                    header_written = True
                    sample_count += SAMPLES_PER_PACKET
                else:
                    print("Timestamps did not change")
except KeyboardInterrupt:
    print("Stopping...")
finally:
    ser.close()
    print(f"Saved {sample_count} samples to {savefile_path}")

Saved 1000 samples to data\06_25_2026\run_15_19_36.csv


## Realtime plot

Reads packets from the Arduino and updates a scrolling view of the four ADC channels

In [6]:
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore

ser = serial.Serial(PORT, BAUD, timeout=1)
channels = ['BPW34_1', 'BPW34_2', 'Inano_1', 'Inano_2']
N = 2000
buffers = {ch: np.zeros(N) for ch in channels}

app = pg.mkQApp("QRNG realtime")
win = pg.GraphicsLayoutWidget(show=True, title="QRNG realtime")
curves = {}
for i, ch in enumerate(channels):
    p = win.ci.addPlot(row=i, col=0, title=ch)
    p.setLabel('left', 'mV')
    curves[ch] = p.plot()


def update():
    if ser.read(1) == b'\xAA' and ser.read(1) == b'\xBB':
        raw = ser.read(BYTES_PER_SAMPLE)
        if len(raw) == BYTES_PER_SAMPLE:
            data = np.frombuffer(raw, dtype=payload_dtype)
            for ch in channels:
                buffers[ch] = np.roll(buffers[ch], -SAMPLES_PER_PACKET)
                buffers[ch][-SAMPLES_PER_PACKET:] = (data[ch] / adc_max * v_ref)*1000 ##from v to mV
                curves[ch].setData(buffers[ch])

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start()
pg.exec()
ser.close()

## Spectrogram Plotting

Make sure to specify load_from_file in the setup block to load from file instead of read real time data

In [7]:
from scipy.signal import get_window

ser = serial.Serial(PORT, BAUD, timeout=1)
NFFT = 4096        
WINDOWSIZE = 2048
WINDOW = 'hamming'
OVERLAP = 90 
BAND = [490, 510]  # bandpass filter range in Hz
FILTER = False

## load from file or read from serial
## after this block: samples[ch] and times are plain 1-D numpy arrays
if load_from_file:
    df = pd.read_csv(load_file_path, index_col='timestamp')
    samples = {ch: df[ch].to_numpy() / adc_max * v_ref for ch in sensors}
    times = df.index.to_numpy()
else:
    sample_chunks = {ch: [] for ch in sensors}
    time_chunks = []
    start = time.time()
    while time.time() - start < 10:
        if ser.read(1) == b'\xAA' and ser.read(1) == b'\xBB':
            raw = ser.read(BYTES_PER_SAMPLE)
            if len(raw) == BYTES_PER_SAMPLE:
                data = np.frombuffer(raw, dtype=payload_dtype)
                time_chunks.append(data['timestamp'])
                for ch in sensors:
                    sample_chunks[ch].append(data[ch] / adc_max * v_ref)
    ser.close()
    samples = {ch: np.concatenate(sample_chunks[ch]) for ch in sensors}
    times = np.concatenate(time_chunks)


Fs = 1.0 / (np.median(np.diff(times)) * 1e-6)
win = get_window(WINDOW, WINDOWSIZE)
noverlap = int(WINDOWSIZE * OVERLAP / 100)

sos = signal.butter(10, BAND, btype='bp', fs=Fs, output='sos') 
filtered = {ch: signal.sosfilt(sos, samples[ch]) for ch in sensors}

with plt.style.context('dark_background'):
    fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True, constrained_layout=True)
    fig.suptitle(f'Channel spectrograms, Sampling Rate = {Fs:.2f} Hz', fontsize=14)

    for ax, ch in zip(axes, sensors):
        if FILTER:
            spec, freqs, t_bins, im = ax.specgram(filtered[ch], Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        else:
            spec, freqs, t_bins, im = ax.specgram(samples[ch], Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        ax.set_ylabel(f'{ch}\n(Hz)')
        ax.set_ylim(0, 600)
        cbar = fig.colorbar(im, ax=ax, pad=0.01)
        cbar.set_label('Power (dB)')

    axes[-1].set_xlabel('Time (s)')
    plt.show()


ValueError: need at least one array to concatenate

## Realtime PSD

In [3]:
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore
from scipy.signal import welch


ser = serial.Serial(PORT, BAUD, timeout=1)

FILTER = True # set to True to enable bandpass filtering
N = 2000
buffers = {ch: np.zeros(N) for ch in sensors}
Fs = 1500  # sampling rate, updated from incoming packets

# per-channel bandpass filter (29.5 - 30.5 Hz), each keeps its own state
sos = signal.butter(4, [495, 505], btype='bp', fs=Fs, output='sos')
sos_zi = {ch: signal.sosfilt_zi(sos) for ch in sensors}

app = pg.mkQApp("QRNG PSD realtime")
win = pg.GraphicsLayoutWidget(show=True, title="QRNG PSD realtime")
curves = {}
plots = []
xticks = [[(1, '1')] + [(f, str(f)) for f in range(100, 1001, 100)]]  # 1, 100, 200, ... 1000
for i, ch in enumerate(sensors):
    p = win.ci.addPlot(row=0, col=i, title=ch)
    p.setLabel('left', 'PSD (mV^2/Hz)')
    p.setLabel('bottom', 'Frequency (Hz)')
    p.getViewBox().setXRange(1, 1000, padding=0)
    p.getViewBox().setYRange(0,2000e-3,padding=0)  # set a minimum y-range to avoid auto-scaling to zero
    p.getAxis('bottom').setTicks(xticks)  # force 1, 100, 200, ... 1000 labels
    if plots:
        p.getViewBox().setYLink(plots[0].getViewBox())  # uniform y axis across all 4 sensors
    plots.append(p)
    curves[ch] = p.plot()


def update():
    if ser.read(1) == b'\xAA' and ser.read(1) == b'\xBB':
        raw = ser.read(BYTES_PER_SAMPLE)
        if len(raw) == BYTES_PER_SAMPLE:
            data = np.frombuffer(raw, dtype=payload_dtype)
            for ch in sensors:
                chunk_clean = (data[ch] / adc_max * v_ref)*1000 ##from v to mV
                # bandpass this chunk, carrying filter state across chunks
                chunk, sos_zi[ch] = signal.sosfilt(sos, chunk_clean, zi=sos_zi[ch])
                buffers[ch] = np.roll(buffers[ch], -SAMPLES_PER_PACKET)
                if FILTER:
                    buffers[ch][-SAMPLES_PER_PACKET:] = chunk
                else:
                    buffers[ch][-SAMPLES_PER_PACKET:] = chunk_clean
                f, pxx = welch(buffers[ch], fs=Fs, nperseg=512)
                curves[ch].setData(f, pxx)

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start()
pg.exec()
ser.close()

## Find the Variance

$f_c$ = 500Hz

The measured voltage over the load resistor in response to an LED at $f_c$: $V(t) = A(t)cos(2 \pi f_c t + \theta) $  

$A(t) = a + am(t) + n(t)$ where

a = the mean of the carrier amplitude, mV

m(t) = the LED relative intensity fluctuation (common to all of the PDS), dimensionless, E[m] = 0

n(t) = the indepedent noise of each PD (shot, thermal, etc) within the band, mV

By using IQ Demodulation, A(t) can be recovered from the carrier wave.

The covariance of all of the A signal pairs from each diode pair can be found as 

$Cov(A_1, A_2) = a_1a_2Var(m)$ since n(t) is uncorrelated and all PDs see the same fractional fluctuation from the LED.  Thus, $Var(m)$ can be isolated as $Var(m) = \frac{Cov(A_1,A_2)}{a_1a_2}$.



Var(m) can be estimated by using the above formula and taking the mean of the most correlated pairs.

Since n(t) is uncorrelated, it can be found that $Var(n_i) = Var(A_i) - a_i^2Var(m)$ for each PD.

In [ ]:
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore
from scipy.signal import welch



ser = serial.Serial(PORT, BAUD, timeout=1)

Fs = 1500 
NUM_SAMPLES_TO_SAVE = 30000   # set to None to run until interrupted
CARRIER_FREQ = 500  # Hz
BAND = [CARRIER_FREQ - 2, CARRIER_FREQ + 2]

filtered_all = {ch: [] for ch in sensors}

sample_count = 0
sos = signal.butter(4, BAND, btype='bp', fs=Fs, output='sos')
sos_zi = {ch: signal.sosfilt_zi(sos) for ch in sensors}

try:
    while NUM_SAMPLES_TO_SAVE is None or sample_count < NUM_SAMPLES_TO_SAVE:
        if ser.read(1) == b'\xAA' and ser.read(1) == b'\xBB':
            raw = ser.read(BYTES_PER_SAMPLE)
            if len(raw) == BYTES_PER_SAMPLE:
                data = np.frombuffer(raw, dtype=payload_dtype)
                for ch in sensors:
                    chunk_raw = (data[ch] / adc_max * v_ref)*1000 ##from v to mV
                    # bandpass this chunk, carrying filter state across chunks
                    chunk_filt, sos_zi[ch] = signal.sosfilt(sos, chunk_raw, zi=sos_zi[ch])
                    filtered_all[ch].append(chunk_filt)
                sample_count += SAMPLES_PER_PACKET
except KeyboardInterrupt:
    print("Stopping...")
finally:
    ser.close()

# skip 200 samples to avoid sos filter transient noise
filt = {ch: np.concatenate(filtered_all[ch])[200:] for ch in sensors}
n = min(len(filt[ch]) for ch in sensors)
t = np.arange(n) / Fs   # time vector in samples, not packets
print(f"Length of time vector: {len(t)}")


# A = demodulated message (mV)
# a = mean message amplitude
# A_i = a_i + b_i * c(t) + n_i   (c = common mode, b_i its per-channel coupling, n_i independent)
FIR_TRANSIENT = 150  # samples to drop after the FIR lowpass settles

A = {}
a = {}
for ch in sensors:
    x = filt[ch][:n]
    I = np.cos(2 * np.pi * CARRIER_FREQ * t) * x
    Q = -np.sin(2 * np.pi * CARRIER_FREQ * t) * x
    I_f, _ = fir_lowpass_filter(I, cutoff_hz=150, fs=Fs)
    Q_f, _ = fir_lowpass_filter(Q, cutoff_hz=150, fs=Fs)
    A[ch] = (2 * np.sqrt(I_f**2 + Q_f**2))[FIR_TRANSIENT:]
    a[ch] = np.mean(A[ch])


X = np.vstack([A[ch] for ch in sensors])
cov_A = np.cov(X)         # mV^2
corr_A = np.corrcoef(X)   # dimensionless [-1, 1]
a_vec = np.array([a[ch] for ch in sensors])
norm_cov = cov_A / np.outer(a_vec, a_vec)   # cov/(a_i a_j); off-diag would be const if single m


def print_matrix(M, title, fmt="{:>12.4f}"):
    print(f"\n{title}")
    print(" " * 10 + "".join(f"{ch:>12}" for ch in sensors))
    for i, ch in enumerate(sensors):
        print(f"{ch:>10}" + "".join(fmt.format(M[i, j]) for j in range(len(sensors))))


print_matrix(cov_A,
    "[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels")
print_matrix(corr_A,
    "[2] CORRELATION (-1..1)-  near 1 = channels track togther (shared LED), 0 = independent")
print_matrix(norm_cov,
    "[3] NORMALIZED COVARIANCE - off-diagonals = var(m) per pair; all equal => one shared LED fluctuation", fmt="{:>12.4e}")

# var(m): LED fluctuation variance as a fraction of signal. Averaged ONLY over
#  pairs that actually share the common mode (|corr| > CORR_GATE), so a
#  decoupled channel (e.g. Inano_2) can't drag the estimate down.
#  Var(A): total variance of each channel's envelope (the diagonal of cov_A).
# % noise: independent noise as a fraction of that channel's total variance,
#  Var(n_i) = Var(A_i) - a_i^2 * var(m).  Only meaningful for channels
#  that are coupled to the LED.
CORR_GATE = 0.5  # only average pairs that actually share the common mode
offdiag = ~np.eye(len(sensors), dtype=bool)
coupled = offdiag & (np.abs(corr_A) > CORR_GATE)
var_m = norm_cov[coupled].mean()
n_pairs = int(coupled.sum() // 2)
print(f"\nvar(m) (mean over {n_pairs} coupled pairs, |corr|>{CORR_GATE}) = {var_m:.4e}   RMS = {np.sqrt(var_m):.4f}")

print(f"\n{'channel':>10}{'Var(A) mV^2':>14}{'Var(n) mV^2':>14}{'% noise':>10}{'coupled':>9}")
for i, ch in enumerate(sensors):
    var_total = cov_A[i, i]
    var_noise = var_total - a_vec[i]**2 * var_m
    pct_noise = 100 * var_noise / var_total
    is_coupled = bool(coupled[i].any())  # shares the common mode with at least one channel
    flag = "yes" if is_coupled else "NO*"
    print(f"{ch:>10}{var_total:>14.4f}{var_noise:>14.4f}{pct_noise:>9.1f}%{flag:>9}")
print("* decoupled channel: a_i^2*var(m) assumption breaks, % noise unreliable")


Length of time vector: 29800

[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels
               BPW34_1     BPW34_2     Inano_1     Inano_2
   BPW34_1      3.5029      5.3767      0.9868     -0.0468
   BPW34_2      5.3767      8.6948      1.8131      0.0061
   Inano_1      0.9868      1.8131      0.5233      0.0394
   Inano_2     -0.0468      0.0061      0.0394      0.0430

[2] CORRELATION (-1..1)-  near 1 = channels track togther (shared LED), 0 = independent
               BPW34_1     BPW34_2     Inano_1     Inano_2
   BPW34_1      1.0000      0.9742      0.7288     -0.1206
   BPW34_2      0.9742      1.0000      0.8500      0.0099
   Inano_1      0.7288      0.8500      1.0000      0.2625
   Inano_2     -0.1206      0.0099      0.2625      1.0000

[3] NORMALIZED COVARIANCE - off-diagonals = var(m) per pair; all equal => one shared LED fluctuation
               BPW34_1     BPW34_2     Inano_1     Inano_2
   B